In [ ]:
!pip uninstall -y langchain langchain-core langchain-community langchain-groq llama-index llama-index-core
!pip cache purge
!pip install -q llama-index llama-index-core llama-index-embeddings-huggingface llama-index-llms-langchain llama-index-readers-file langchain-community langchain-groq pypdf nest_asyncio
!pip install -q llama-index llama-index-core llama-index-embeddings-huggingface llama-index-llms-langchain llama-index-readers-file langchain-community langchain-groq pypdf nest_asyncio openpyxl

Found existing installation: langchain 1.3.1
Uninstalling langchain-1.3.1:
  Successfully uninstalled langchain-1.3.1
Found existing installation: langchain-core 1.4.0
Uninstalling langchain-core-1.4.0:
  Successfully uninstalled langchain-core-1.4.0
Found existing installation: langchain-community 0.4.1
Uninstalling langchain-community-0.4.1:
  Successfully uninstalled langchain-community-0.4.1
Found existing installation: langchain-groq 1.1.2
Uninstalling langchain-groq-1.1.2:
  Successfully uninstalled langchain-groq-1.1.2
Found existing installation: llama-index 0.14.22
Uninstalling llama-index-0.14.22:
  Successfully uninstalled llama-index-0.14.22
Found existing installation: llama-index-core 0.14.22
Uninstalling llama-index-core-0.14.22:
  Successfully uninstalled llama-index-core-0.14.22
Files removed: 36
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [ ]:
import os
import nest_asyncio
from google.colab import userdata
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from langchain_groq import ChatGroq

nest_asyncio.apply()

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print("[INFO] Memuat HuggingFace Embedding Lokal (BAAI/bge-small-en-v1.5)...")

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)
print("[INFO] Menghubungkan ke Groq Cloud LLM (llama-3.1-8b-instant)...")

Settings.llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model_name="llama-3.1-8b-instant",
    temperature=0
)
print("\n🎉 [SUKSES] Seluruh konfigurasi model siap digunakan!")

[INFO] Memuat HuggingFace Embedding Lokal (BAAI/bge-small-en-v1.5)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] Menghubungkan ke Groq Cloud LLM (llama-3.1-8b-instant)...

🎉 [SUKSES] Seluruh konfigurasi model siap digunakan!


In [ ]:
import os
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

if not os.path.exists("data") or len(os.listdir("data")) == 0:
    os.makedirs("data", exist_ok=True)
    print("PERINGATAN: Folder 'data' kosong atau tidak ditemukan.")
    print("Silakan upload file PDF Anda ke dalam folder 'data' di panel kiri Google Colab terlebih dahulu, lalu jalankan ulang cell ini.")
else:
    print("[INFO] Membaca file PDF dari folder 'data'...")
    documents = SimpleDirectoryReader("data").load_data()
    print(f"[INFO] Sukses membaca dokumen. Total: {len(documents)} halaman terdeteksi.")
    print("[INFO] Membangun Vector Index di latar belakang...")

    index = VectorStoreIndex.from_documents(documents)
    query_engine = index.as_query_engine(similarity_top_k=3)
    print("[INFO] Sistem RAG Berhasil Diaktifkan!")
    print("\n" + "="*50)
    print("🤖 CHATBOT RAG DOKUMEN AKTIF")
    print("Ketik pertanyaan Anda lalu tekan ENTER.")
    print("Ketik 'exit' untuk menyudahi percakapan.")
    print("="*50 + "\n")

    while True:
        question = input("Kamu: ").strip()
        if question.lower() == 'exit':
            print("Sesi chat diakhiri. Sampai jumpa!")
            break
        if not question:
            continue
        print("\n[Sistem]: Sedang mencari referensi di PDF dan merumuskan jawaban...")
        try:
            response = query_engine.query(question)
            print(f"\nBot: {response}")
            print("\n" + "-"*40 + "\n")
        except Exception as e:
            print(f"\n[ERROR]: Terjadi kendala komunikasi dengan API: {e}\n")

[INFO] Membaca file PDF dari folder 'data'...
[INFO] Sukses membaca dokumen. Total: 14 halaman terdeteksi.
[INFO] Membangun Vector Index di latar belakang...
[INFO] Sistem RAG Berhasil Diaktifkan!

🤖 CHATBOT RAG DOKUMEN AKTIF
Ketik pertanyaan Anda lalu tekan ENTER.
Ketik 'exit' untuk menyudahi percakapan.

Kamu: apa isi dari article 1?

[Sistem]: Sedang mencari referensi di PDF dan merumuskan jawaban...

Bot: Article 1 menjelaskan tentang struktur dan kekuasaan Kongres Amerika Serikat. Kongres terdiri dari dua bagian, yaitu Senat dan Dewan Perwakilan Rakyat.

----------------------------------------

Kamu: apa tujuan dari article 2?

[Sistem]: Sedang mencari referensi di PDF dan merumuskan jawaban...

Bot: Tujuan dari Article 2 adalah untuk menentukan hak pilih warga negara Amerika Serikat dalam pemilihan presiden, wakil presiden, dan anggota Kongres.

----------------------------------------

Kamu: apa kesimpulan dari semua article yang ada?

[Sistem]: Sedang mencari referensi di PDF 